In [ ]:
import plotly.express as px
import polars as pl
from folium.plugins import FastMarkerCluster

from fryer import all as fryer

In [ ]:
df_postcode = fryer.data.uk_gov_ons_postcode_directory.read().select(
    pl.col("postcode"),
    pl.col("longitude"),
    pl.col("latitude"),
)
df_postcode.tail().collect()

In [ ]:
df = fryer.data.uk_gov_compare_school_performance.read_raw(year=2023).join(
    df_postcode, on="postcode", how="left"
)

df.head().collect()

In [ ]:
# Postcodes where the join has failed
df.filter(pl.col("longitude").is_null()).collect()["postcode"].value_counts().sort(
    "count", descending=True
)

In [ ]:
for col in (
    "status",
    "group",
    "funding_type",
    "ofsted_rating",
    "religious_character",
    "gender",
):
    df_len = df.group_by([col]).len().collect()
    display(
        df_len.pipe(
            px.bar,
            x=col,
            y="len",
            color=col,
            title=col,
            category_orders={
                col: df_len.sort(by="len", descending=True)[col].to_list(),
            },
        ),
    )

In [ ]:
df_interested = df.filter(pl.col("status") == "Open").collect()

df_interested.columns

In [ ]:
uk_map_primary = fryer.map.create_uk()

df_primary = (
    df_interested.filter(pl.col("is_primary"))
    .drop_nulls(subset=["latitude", "longitude"])
    .with_columns(
        (
            pl.when(
                (~pl.col("ofsted_rating").is_in(["Good", "Outstanding"]))
                .or_(pl.col("gender").is_in(["Boys"]))
                .or_(
                    ~pl.col("funding_type").is_in(
                        ["State-funded primary", "State-funded secondary"]
                    )
                )
            )
            .then(pl.lit("red"))
            .when(pl.col("ofsted_rating").is_in(["Good"]))
            .then(pl.lit("blue"))
            .when(pl.col("ofsted_rating").is_in(["Outstanding"]))
            .then(pl.lit("green"))
            .alias("color")
        ),
        pl.selectors.string().fill_null("NA"),
        pl.selectors.numeric().fill_null(float("nan")),
    )
    .with_columns(
        (
            pl.col("name")
            + "<br>Type: "
            + pl.col("funding_type")
            + "<br>Gender: "
            + pl.col("gender")
            + "<br>Ofsted Rating: "
            + pl.col("ofsted_rating")
            + " @ "
            + pl.col("date_last_ofsted_inspection")
            .dt.strftime("%Y-%m-%d")
            .fill_null("NA")
            + "<br>Religious Character: "
            + pl.col("religious_character")
        ).alias("tooltip"),
    )
    .with_columns(
        (
            pl.col("tooltip")
            + "<br>Admissions Policy: "
            + pl.col("admissions_policy")
            + "<br>Number of Pupils: "
            + pl.col("number_of_pupils").cast(pl.String)
            + "<br>Girls: "
            + pl.col("percent_girls").round(2).cast(pl.String)
            + "<br>Age Low: "
            + pl.col("age_low").cast(pl.String)
            + ", Age High: "
            + pl.col("age_high").cast(pl.String)
            + "<br>Absence: "
            + pl.col("percent_absence").round(2).cast(pl.String)
            + ", Persistent Absence: "
            + pl.col("percent_persistent_absence").round(2).cast(pl.String)
            + "<br>First Language English: "
            + pl.col("percent_first_language_english").round(2).cast(pl.String)
            + "<br>Free School Meals: "
            + pl.col("percent_free_school_meals").round(2).cast(pl.String)
            + ", Last 6 Years: "
            + pl.col("percent_free_school_meals_last_6_years").round(2).cast(pl.String)
            + "<br>Education Health Care Plan: "
            + pl.col("percent_education_health_care_plan").round(2).cast(pl.String)
            + "<br>Special Education Needs Support: "
            + pl.col("percent_special_education_needs_support").round(2).cast(pl.String)
        ).alias("popup")
    )
)
marker_cluster = FastMarkerCluster(
    data=df_primary[["latitude", "longitude", "color", "tooltip", "popup"]].rows(),
    callback="""\
function (row) {
    var icon, marker;
    icon = L.AwesomeMarkers.icon({
        icon: "map-marker", markerColor: row[2]});
    marker = L.marker(new L.LatLng(row[0], row[1]));
    marker.setIcon(icon);
    // add tooltip
    var tooltip = L.tooltip();
    var tooltip_text = $(`<div id='mytext' class='display_text' style='width: 100.0%; height: 100.0%;'> ${row[3]}</div>`)[0];
    tooltip.setContent(tooltip_text);
    marker.bindTooltip(tooltip);
    // add popup
    var popup = L.popup({maxWidth: '300'});
    var popup_text = $(`<div id='mytext' class='display_text' style='width: 100.0%; height: 100.0%;'> ${row[4]}</div>`)[0];
    popup.setContent(popup_text);
    marker.bindPopup(popup);
    return marker;
};
""",
).add_to(uk_map_primary)
# marker_cluster = MarkerCluster().add_to(uk_map_primary)
# for data in df_primary.iter_rows(named=True):
#     folium.Marker(
#         location=[data["latitude"], data["longitude"]],
#         tooltip=data["name"],
#         popup=None,
#         icon=folium.Icon(color=data["color"])
#     ).add_to(marker_cluster)

uk_map_primary